# Phase 7: ViSNet 3D Pre-training (300k, Colab A100)
Equivariant vector-scalar GNN on ETKDG 3D graphs.
Outputs: `visnet_3d_encoder.pt` + `visnet_3d_best.pt` + metrics

In [ ]:
!pip install -q torch-geometric optuna
!pip install -q pyg-lib -f https://data.pyg.org/whl/torch-$(python -c "import torch; print(torch.__version__.split('+')[0])")+cu$(python -c "import torch; print(torch.version.cuda.replace('.',''))").html

In [ ]:
import torch, os, json, time
import numpy as np
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_mean_pool
from torch_geometric.nn.models.visnet import ViSNetBlock
import optuna

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
props = torch.cuda.get_device_properties(0)
print(f'{props.name} | {props.total_memory/1e9:.1f} GB')

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/molgap'
SAVE_DIR = f'{DRIVE_DIR}/phase7_results'
os.makedirs(SAVE_DIR, exist_ok=True)
print('Save dir:', SAVE_DIR)

In [ ]:
# Load 3D graphs
GRAPH_PATH = f'{DRIVE_DIR}/pyg_3d_graphs_etkdg_300k.pt'
graphs = torch.load(GRAPH_PATH, weights_only=False)
print(f'Loaded {len(graphs)} graphs')
print(f'Gasteiger charges: {hasattr(graphs[0], "charges")}')

# Train/val/test split 8:1:1
N = len(graphs)
idx = np.random.RandomState(42).permutation(N)
n_train = int(0.8 * N)
n_val = int(0.1 * N)
train_set = [graphs[i] for i in idx[:n_train]]
val_set = [graphs[i] for i in idx[n_train:n_train+n_val]]
test_set = [graphs[i] for i in idx[n_train+n_val:]]
print(f'Split: train={len(train_set)}, val={len(val_set)}, test={len(test_set)}')

In [ ]:
import torch.nn as nn

class ViSNetWrapper(nn.Module):
    def __init__(self, hidden_channels=128, num_layers=6, num_rbf=32,
                 cutoff=5.0, num_heads=8, max_num_neighbors=32,
                 lmax=1, trainable_rbf=False, dropout=0.1, n_targets=3):
        super().__init__()
        self.encoder = ViSNetBlock(
            lmax=lmax, num_heads=num_heads, num_layers=num_layers,
            hidden_channels=hidden_channels, num_rbf=num_rbf,
            trainable_rbf=trainable_rbf, cutoff=cutoff,
            max_num_neighbors=max_num_neighbors,
        )
        self.hidden_channels = hidden_channels
        self.head = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, hidden_channels // 2),
            nn.SiLU(),
            nn.Linear(hidden_channels // 2, n_targets),
        )

    def forward(self, z, pos, batch):
        return self.head(self.encode(z, pos, batch))

    def encode(self, z, pos, batch):
        x, _v = self.encoder(z, pos, batch)
        return global_mean_pool(x, batch)

print('ViSNetWrapper defined')

In [ ]:
def run_training(params, train_set, val_set, max_epochs=80,
                 patience=15, resume_ckpt=None, save_prefix='visnet'):
    model = ViSNetWrapper(
        hidden_channels=params['hidden_channels'],
        num_layers=params['num_layers'],
        num_rbf=params['num_rbf'],
        cutoff=params['cutoff'],
        num_heads=params['num_heads'],
        dropout=params['dropout'],
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=params['lr'],
                                  weight_decay=params['weight_decay'])

    if params['scheduler'] == 'plateau':
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, patience=5, factor=0.5, min_lr=1e-6)
    else:
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=max_epochs, eta_min=1e-6)

    scaler = torch.amp.GradScaler()
    criterion = nn.L1Loss()

    start_epoch = 0
    if resume_ckpt and os.path.exists(resume_ckpt):
        ckpt = torch.load(resume_ckpt, weights_only=False)
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        sched.load_state_dict(ckpt['scheduler'])
        scaler.load_state_dict(ckpt['scaler'])
        start_epoch = ckpt['epoch'] + 1
        print(f'Resumed from epoch {start_epoch}')

    bs = params['batch_size']
    NUM_WORKERS = 4
    train_loader = DataLoader(train_set, batch_size=bs, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_set, batch_size=bs, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)

    best_val = float('inf')
    wait = 0

    for epoch in range(start_epoch, max_epochs):
        model.train()
        t0 = time.time()
        train_loss = 0.0
        for batch_data in train_loader:
            batch_data = batch_data.to(device)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                pred = model(batch_data.z, batch_data.pos, batch_data.batch)
                loss = criterion(pred, batch_data.y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item() * batch_data.num_graphs
        train_loss /= len(train_set)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_data in val_loader:
                batch_data = batch_data.to(device)
                with torch.amp.autocast('cuda'):
                    pred = model(batch_data.z, batch_data.pos, batch_data.batch)
                    loss = criterion(pred, batch_data.y)
                val_loss += loss.item() * batch_data.num_graphs
        val_loss /= len(val_set)

        if params['scheduler'] == 'plateau':
            sched.step(val_loss)
        else:
            sched.step()

        elapsed = time.time() - t0
        lr_now = optimizer.param_groups[0]['lr']
        print(f'  ep{epoch:03d} train={train_loss:.4f} val={val_loss:.4f} '
              f'lr={lr_now:.2e} {elapsed:.0f}s')

        if val_loss < best_val:
            best_val = val_loss
            wait = 0
            torch.save(model.state_dict(),
                       f'{SAVE_DIR}/{save_prefix}_best.pt')
            torch.save({'hidden_channels': params['hidden_channels'],
                        'num_layers': params['num_layers'],
                        'embedding': model.encoder.state_dict()},
                       f'{SAVE_DIR}/{save_prefix}_encoder.pt')
        else:
            wait += 1

        if epoch % 5 == 0:
            torch.save({
                'epoch': epoch, 'model': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'scheduler': sched.state_dict(),
                'scaler': scaler.state_dict(),
                'best_val': best_val,
            }, f'{SAVE_DIR}/{save_prefix}_ckpt.pt')

        if wait >= patience:
            print(f'  Early stop at epoch {epoch}')
            break

    return best_val

print('run_training defined')

In [ ]:
DB_PATH = f'sqlite:///{SAVE_DIR}/optuna_visnet_3d.db'

def objective(trial):
    params = {
        'hidden_channels': trial.suggest_categorical('hidden_channels', [128, 192, 256]),
        'num_layers': trial.suggest_int('num_layers', 4, 8),
        'num_rbf': trial.suggest_categorical('num_rbf', [32, 50, 64]),
        'cutoff': trial.suggest_float('cutoff', 5.0, 10.0, step=1.0),
        'num_heads': trial.suggest_categorical('num_heads', [4, 8]),
        'dropout': trial.suggest_float('dropout', 0.0, 0.2, step=0.05),
        'lr': trial.suggest_float('lr', 1e-5, 5e-4, log=True),
        'weight_decay': trial.suggest_float('weight_decay', 1e-7, 1e-4, log=True),
        'batch_size': trial.suggest_categorical('batch_size', [64, 128, 256]),
        'scheduler': trial.suggest_categorical('scheduler', ['plateau', 'cosine']),
    }
    print(f'\nTrial {trial.number} | h={params["hidden_channels"]} '
          f'L={params["num_layers"]} cut={params["cutoff"]} '
          f'lr={params["lr"]:.2e} bs={params["batch_size"]}')

    ckpt_path = f'{SAVE_DIR}/visnet_trial{trial.number}_ckpt.pt'
    val_mae = run_training(params, train_set, val_set,
                           max_epochs=80, patience=15,
                           resume_ckpt=ckpt_path,
                           save_prefix=f'visnet_trial{trial.number}')
    return val_mae

N_TRIALS = 20
study = optuna.create_study(
    study_name='visnet_3d_300k',
    storage=DB_PATH,
    direction='minimize',
    load_if_exists=True,
)
done = len(study.trials)
remaining = N_TRIALS - done
print(f'Optuna: {done} trials done, {remaining} remaining')
if remaining > 0:
    study.optimize(objective, n_trials=remaining)

In [ ]:
# Retrain best trial on full epochs
print('\n=== Best Trial ===')
bp = study.best_params
print(f'MAE={study.best_value:.4f}')
print(bp)

print('\nRetraining best config with more patience...')
best_val = run_training(bp, train_set, val_set,
                        max_epochs=150, patience=25,
                        save_prefix='visnet_3d_best')

In [ ]:
# Test evaluation
bp = study.best_params
model = ViSNetWrapper(
    hidden_channels=bp['hidden_channels'],
    num_layers=bp['num_layers'],
    num_rbf=bp['num_rbf'],
    cutoff=bp['cutoff'],
    num_heads=bp['num_heads'],
    dropout=bp['dropout'],
).to(device)
model.load_state_dict(torch.load(f'{SAVE_DIR}/visnet_3d_best_best.pt',
                                  weights_only=False))
model.eval()

test_loader = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=4)
from sklearn.metrics import mean_absolute_error, r2_score

all_pred, all_true = [], []
with torch.no_grad():
    for batch_data in test_loader:
        batch_data = batch_data.to(device)
        pred = model(batch_data.z, batch_data.pos, batch_data.batch)
        all_pred.append(pred.cpu().numpy())
        all_true.append(batch_data.y.cpu().numpy())

all_pred = np.concatenate(all_pred)
all_true = np.concatenate(all_true)

labels = ['HOMO', 'LUMO', 'Gap']
metrics = {}
for i, name in enumerate(labels):
    mae = mean_absolute_error(all_true[:, i], all_pred[:, i])
    r2 = r2_score(all_true[:, i], all_pred[:, i])
    print(f'{name}: MAE={mae:.4f} eV, R2={r2:.4f}')
    metrics[name] = {'mae': float(mae), 'r2': float(r2)}

metrics['best_params'] = bp
metrics['best_val_mae'] = float(study.best_value)
with open(f'{SAVE_DIR}/visnet_3d_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'\nMetrics saved to {SAVE_DIR}/visnet_3d_metrics.json')

In [ ]:
# Save encoder embeddings for fusion
print('Extracting embeddings for fusion...')
all_loader = DataLoader(graphs, batch_size=256, shuffle=False, num_workers=4)
embeddings_3d = []
with torch.no_grad():
    for batch_data in all_loader:
        batch_data = batch_data.to(device)
        emb = model.encode(batch_data.z, batch_data.pos, batch_data.batch)
        embeddings_3d.append(emb.cpu())
embeddings_3d = torch.cat(embeddings_3d, dim=0)
print(f'3D embeddings: {embeddings_3d.shape}')
torch.save(embeddings_3d, f'{SAVE_DIR}/visnet_3d_embeddings.pt')
print('Done! Download visnet_3d_embeddings.pt for fusion.')